# v3.3 — sinus takikardia asli, aturan produk, abstain, dan metrik episode

Evaluasi saja (tanpa training; paket produksi tidak disentuh). Menindaklanjuti riset 6 November 2026:
1. alarm palsu pada **EKG latihan fisik asli** (EPFL, Wrist, EPHNOGRAM) lewat pipeline perangkat dan lewat R-peak manual;
2. aturan produk v3.2 ("≥ 110 bpm & CV IBI ≤ 0,05 → takikardia teratur, perlu dicek") di data asli;
3. keluaran abstain dengan conformal prediction (LAC, cakupan 90%);
4. metrik episode gaya ANSI/AAMI EC57 di AFDB.
Logika di `takikardia_asli.py`. Dijalankan dengan cwd = folder ini (`python tools/run_notebook.py v3.3_takikardia_asli/takikardia_asli.ipynb`).

In [1]:
import json, sys, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path[:0] = ["../v11_model_tim"]
import takikardia_asli as K
from bersama import evaluasi as Ev
p_j = K.CACHE / "jendela.npz"
J = K.muat_jendela() if p_j.exists() else K.bangun_jendela()
ok = J["ok"]
contoh = [r for i, r in enumerate(K.rekaman_epfl()) if i < 3] + [r for i, r in enumerate(K.rekaman_wrist()) if i < 3]
print("bukti logika window identik dengan hrv_core.signal_to_windows:", [K.bukti_window(r) for r in contoh])

bukti logika window identik dengan hrv_core.signal_to_windows: [2, 2, 2, 58, 57, 58]


## 1. Data latihan fisik

In [2]:
rows = {}
for s in ("EPFL", "Wrist", "EPHNOGRAM"):
    for mode in ("perangkat", "manual"):
        m = (J["sumber"] == s) & (J["mode"] == mode)
        if m.any():
            mo = m & ok
            rows[f"{s} / {mode}"] = {"subjek": int(np.unique(J["subjek"][m]).size), "window": int(m.sum()), "lolos cek": int(mo.sum()),
                                     "% lolos": round(100 * mo.sum() / m.sum(), 1), "detak median": round(float(np.nanmedian(J["hr"][mo])), 0),
                                     "detak maks": round(float(np.nanmax(J["hr"][mo])), 0), "CV median": round(float(np.nanmedian(J["cv"][mo])), 3),
                                     "≥ 120 bpm": int((mo & (J["hr"] >= 120)).sum())}
data_tab = pd.DataFrame(rows).T
data_tab

,subjek,window,lolos cek,% lolos,detak median,detak maks,CV median,≥ 120 bpm
EPFL / perangkat,20.0,198.0,135.0,68.2,165.0,196.0,0.028,133.0
EPFL / manual,20.0,198.0,188.0,94.9,173.0,197.0,0.009,188.0
Wrist / perangkat,8.0,670.0,560.0,83.6,98.0,130.0,0.066,66.0
Wrist / manual,8.0,670.0,643.0,96.0,98.0,179.0,0.033,114.0
EPHNOGRAM / perangkat,24.0,11004.0,10127.0,92.0,100.0,196.0,0.047,1990.0


## 2. Prediksi semua model (paket v1/v2/v3.1; model fold v7, v8, v11 dengan bias fold masing-masing)

In [3]:
p_p = K.CACHE / "prediksi.npz"
if p_p.exists() and np.load(p_p)["v31"].shape[1] == ok.sum():
    z = np.load(p_p); hasil = {k: z[k] for k in z.files if k != "p_v31"}; p31 = z["p_v31"]
else:
    hasil, p31 = K.prediksi_semua(J, device="cuda")
MODEL = ["v1", "v2_prod", "v31"] + list(K.KELUARGA)
LABEL = {"v1": "v1 paket", "v2_prod": "v2 paket", "v31": "v3.1 paket (produksi)"} | {k: v["label"] for k, v in K.KELUARGA.items()}
print({k: hasil[k].shape for k in MODEL})

{'v1': (1, 11653), 'v2_prod': (1, 11653), 'v31': (1, 11653), 'v7_130': (25, 11653), 'v7_160': (25, 11653), 'v8_8a': (25, 11653), 'v8_8c': (25, 11653), 'v11_bigru': (25, 11653), 'v11_bigru_wp100': (25, 11653), 'v11_tcn_wp100': (25, 11653)}


In [4]:
alarm = {}
for mode in ("perangkat", "manual"):
    for s in ("EPFL", "Wrist", "EPHNOGRAM"):
        mask = (J["sumber"][ok] == s) & (J["mode"][ok] == mode)
        if mask.sum() == 0:
            continue
        alarm[f"{s}/{mode}"] = {k: K.alarm_per_pita(J, hasil[k], mask) for k in MODEL}
def tabel(kunci):
    return pd.DataFrame({LABEL[k]: {b: f"{v['alarm']*100:.0f}% (n={v['n']})" for b, v in alarm[kunci][k].items()} for k in MODEL}).T
for kunci in alarm:
    print("==", kunci); display(tabel(kunci))

== EPFL/perangkat


,0–100 bpm,130–140 bpm,140–150 bpm,150–∞ bpm
v1 paket,0% (n=2),100% (n=3),100% (n=17),100% (n=113)
v2 paket,100% (n=2),100% (n=3),100% (n=17),100% (n=113)
v3.1 paket (produksi),0% (n=2),100% (n=3),100% (n=17),100% (n=113)
v7a: + sinus takikardia sintetis 100–130 bpm,66% (n=2),100% (n=3),100% (n=17),100% (n=113)
v7b: + sinus takikardia sintetis 100–160 bpm,94% (n=2),100% (n=3),96% (n=17),45% (n=113)
v8a: AFDB + Chapman (termasuk ST asli),82% (n=2),100% (n=3),80% (n=17),38% (n=113)
v8c: AFDB + Chapman tanpa label AFL,84% (n=2),97% (n=3),83% (n=17),21% (n=113)
v11 BiGRU bawaan,44% (n=2),99% (n=3),100% (n=17),100% (n=113)
v11 BiGRU bobot penuh,66% (n=2),100% (n=3),100% (n=17),100% (n=113)
v11 TCN bobot penuh,80% (n=2),100% (n=3),100% (n=17),100% (n=113)


== Wrist/perangkat


,0–100 bpm,100–110 bpm,110–120 bpm,120–130 bpm,130–140 bpm
v1 paket,32% (n=305),37% (n=108),93% (n=81),98% (n=64),100% (n=2)
v2 paket,48% (n=305),35% (n=108),77% (n=81),100% (n=64),100% (n=2)
v3.1 paket (produksi),29% (n=305),22% (n=108),63% (n=81),100% (n=64),100% (n=2)
v7a: + sinus takikardia sintetis 100–130 bpm,25% (n=305),19% (n=108),30% (n=81),8% (n=64),78% (n=2)
v7b: + sinus takikardia sintetis 100–160 bpm,48% (n=305),35% (n=108),48% (n=81),50% (n=64),78% (n=2)
v8a: AFDB + Chapman (termasuk ST asli),34% (n=305),23% (n=108),33% (n=81),46% (n=64),98% (n=2)
v8c: AFDB + Chapman tanpa label AFL,26% (n=305),21% (n=108),39% (n=81),52% (n=64),78% (n=2)
v11 BiGRU bawaan,22% (n=305),16% (n=108),34% (n=81),83% (n=64),90% (n=2)
v11 BiGRU bobot penuh,30% (n=305),21% (n=108),48% (n=81),94% (n=64),100% (n=2)
v11 TCN bobot penuh,22% (n=305),15% (n=108),31% (n=81),73% (n=64),100% (n=2)


== EPHNOGRAM/perangkat


,0–100 bpm,100–110 bpm,110–120 bpm,120–130 bpm,130–140 bpm,140–150 bpm,150–∞ bpm
v1 paket,31% (n=5034),25% (n=1862),84% (n=1241),95% (n=813),99% (n=439),100% (n=278),100% (n=460)
v2 paket,44% (n=5034),28% (n=1862),84% (n=1241),100% (n=813),100% (n=439),100% (n=278),100% (n=460)
v3.1 paket (produksi),21% (n=5034),16% (n=1862),62% (n=1241),97% (n=813),99% (n=439),99% (n=278),100% (n=460)
v7a: + sinus takikardia sintetis 100–130 bpm,22% (n=5034),13% (n=1862),17% (n=1241),23% (n=813),77% (n=439),99% (n=278),100% (n=460)
v7b: + sinus takikardia sintetis 100–160 bpm,40% (n=5034),25% (n=1862),38% (n=1241),64% (n=813),90% (n=439),96% (n=278),57% (n=460)
v8a: AFDB + Chapman (termasuk ST asli),25% (n=5034),18% (n=1862),24% (n=1241),52% (n=813),81% (n=439),94% (n=278),58% (n=460)
v8c: AFDB + Chapman tanpa label AFL,18% (n=5034),15% (n=1862),28% (n=1241),53% (n=813),82% (n=439),95% (n=278),41% (n=460)
v11 BiGRU bawaan,22% (n=5034),12% (n=1862),29% (n=1241),80% (n=813),96% (n=439),99% (n=278),99% (n=460)
v11 BiGRU bobot penuh,22% (n=5034),15% (n=1862),41% (n=1241),91% (n=813),99% (n=439),100% (n=278),100% (n=460)
v11 TCN bobot penuh,20% (n=5034),10% (n=1862),24% (n=1241),74% (n=813),97% (n=439),100% (n=278),100% (n=460)


== EPFL/manual


,130–140 bpm,140–150 bpm,150–∞ bpm
v1 paket,100% (n=2),100% (n=18),100% (n=168)
v2 paket,100% (n=2),100% (n=18),100% (n=168)
v3.1 paket (produksi),100% (n=2),100% (n=18),100% (n=168)
v7a: + sinus takikardia sintetis 100–130 bpm,100% (n=2),100% (n=18),100% (n=168)
v7b: + sinus takikardia sintetis 100–160 bpm,100% (n=2),96% (n=18),29% (n=168)
v8a: AFDB + Chapman (termasuk ST asli),94% (n=2),79% (n=18),19% (n=168)
v8c: AFDB + Chapman tanpa label AFL,86% (n=2),86% (n=18),11% (n=168)
v11 BiGRU bawaan,100% (n=2),100% (n=18),100% (n=168)
v11 BiGRU bobot penuh,100% (n=2),100% (n=18),100% (n=168)
v11 TCN bobot penuh,100% (n=2),100% (n=18),100% (n=168)


== Wrist/manual


,0–100 bpm,100–110 bpm,110–120 bpm,120–130 bpm,130–140 bpm,150–∞ bpm
v1 paket,19% (n=349),27% (n=94),98% (n=86),100% (n=107),100% (n=5),100% (n=2)
v2 paket,30% (n=349),15% (n=94),91% (n=86),100% (n=107),100% (n=5),100% (n=2)
v3.1 paket (produksi),9% (n=349),4% (n=94),58% (n=86),100% (n=107),100% (n=5),100% (n=2)
v7a: + sinus takikardia sintetis 100–130 bpm,9% (n=349),4% (n=94),6% (n=86),4% (n=107),54% (n=5),100% (n=2)
v7b: + sinus takikardia sintetis 100–160 bpm,31% (n=349),10% (n=94),22% (n=86),50% (n=107),81% (n=5),40% (n=2)
v8a: AFDB + Chapman (termasuk ST asli),9% (n=349),6% (n=94),15% (n=86),42% (n=107),74% (n=5),20% (n=2)
v8c: AFDB + Chapman tanpa label AFL,8% (n=349),5% (n=94),19% (n=86),48% (n=107),71% (n=5),0% (n=2)
v11 BiGRU bawaan,10% (n=349),4% (n=94),24% (n=86),85% (n=107),99% (n=5),100% (n=2)
v11 BiGRU bobot penuh,9% (n=349),6% (n=94),39% (n=86),95% (n=107),99% (n=5),100% (n=2)
v11 TCN bobot penuh,8% (n=349),3% (n=94),19% (n=86),73% (n=107),96% (n=5),100% (n=2)


In [5]:
# ringkasan utama: semua sumber, mode perangkat, window ≥ 120 bpm; CI bootstrap subjek; berpasangan vs v3.1
mask_dev = J["mode"][ok] == "perangkat"
hr_ok, subj_ok = J["hr"][ok], J["subjek"][ok]
utama = {}
for k in MODEL:
    m = mask_dev & (hr_ok >= 120)
    t, c, _ = K._boot_subjek(subj_ok[m], (hasil[k][:, m] != 0).mean(0))
    b = K.banding_alarm(J, hasil[k], hasil["v31"], mask_dev) if k != "v31" else None
    utama[k] = dict(alarm=t, ci=c.tolist(), n=int(m.sum()), subjek=int(np.unique(subj_ok[m]).size), banding=b)
pd.DataFrame({LABEL[k]: {"alarm ≥120 bpm": f"{v['alarm']*100:.1f}% [{v['ci'][0]*100:.0f}–{v['ci'][1]*100:.0f}]",
                         "Δ vs v3.1": "" if v["banding"] is None else f"{v['banding']['delta']*100:+.1f} poin, P(lebih sedikit) = {v['banding']['p_kurang']:.3f}"}
              for k, v in utama.items()}).T

,alarm ≥120 bpm,Δ vs v3.1
v1 paket,98.0% [96–100],"-0.4 poin, P(lebih sedikit) = 0.887"
v2 paket,99.9% [100–100],"+1.5 poin, P(lebih sedikit) = 0.000"
v3.1 paket (produksi),98.4% [97–100],
v7a: + sinus takikardia sintetis 100–130 bpm,63.9% [58–70],"-34.5 poin, P(lebih sedikit) = 1.000"
v7b: + sinus takikardia sintetis 100–160 bpm,70.9% [66–75],"-27.5 poin, P(lebih sedikit) = 1.000"
v8a: AFDB + Chapman (termasuk ST asli),63.8% [57–71],"-34.6 poin, P(lebih sedikit) = 1.000"
v8c: AFDB + Chapman tanpa label AFL,60.3% [53–68],"-38.1 poin, P(lebih sedikit) = 1.000"
v11 BiGRU bawaan,91.0% [87–94],"-7.4 poin, P(lebih sedikit) = 1.000"
v11 BiGRU bobot penuh,96.0% [94–98],"-2.4 poin, P(lebih sedikit) = 1.000"
v11 TCN bobot penuh,88.9% [85–92],"-9.5 poin, P(lebih sedikit) = 1.000"


## 3. Aturan produk dan abstain conformal

In [6]:
d32 = K.T.load()
pred31 = np.stack([K.cv.crossfit_bias(d32["y"], d32["probs"]["v31"][r], d32["folds"][r])[0].argmax(1) for r in range(5)])
aturan_afdb = K.T.product_rule(d32, pred31)["≥110 bpm & CV ≤ 0.05"]
rule = K.kena_aturan(hr_ok, J["cv"][ok])
k31 = hasil["v31"][0]
aturan_asli = {}
for s in ("EPFL", "Wrist", "EPHNOGRAM"):
    for mode in ("perangkat", "manual"):
        m = (J["sumber"][ok] == s) & (J["mode"][ok] == mode) & (hr_ok >= 120)
        if m.sum():
            aturan_asli[f"{s}/{mode}"] = dict(n=int(m.sum()), alarm_paket=float((k31[m] != 0).mean()), kena_aturan=float(rule[m].mean()),
                                              alarm_sisa=float(((k31[m] != 0) & ~rule[m]).mean()))
print("biaya di AFDB (v3.2):", {k: round(v, 3) for k, v in aturan_afdb.items()})
pd.DataFrame(aturan_asli).T.round(3)

biaya di AFDB (v3.2): {'Normal kena aturan': 0.001, 'AFIB kena aturan': 0.002, 'AFL kena aturan': 0.567, 'alarm palsu dialihkan': 0.009, 'alarm benar dialihkan': 0.012, 'AFL terdeteksi dialihkan': 0.86}


,n,alarm_paket,kena_aturan,alarm_sisa
EPFL/perangkat,133.0,1.000,0.729,0.271
EPFL/manual,188.0,1.000,0.968,0.032
Wrist/perangkat,66.0,1.000,0.833,0.167
Wrist/manual,114.0,1.000,0.851,0.149
EPHNOGRAM/perangkat,1990.0,0.982,0.613,0.370


In [7]:
abst = {}
for kosong in (False, True):
    dd, dec, q = K.abstain_afdb(kosong_ragu=kosong)
    abst[f"kosong_ragu={kosong}"] = {"ambang_median": float(np.median(q)),
        "afdb": {K.CLASSES[c]: [float((dec[:, dd["y"] == c] == j).mean()) for j in range(4)] for c in range(3)}}
q_pk = K.ambang_paket()
for kosong in (False, True):
    dr = K.keputusan_abstain(p31, q_pk, kosong)
    m = mask_dev & (hr_ok >= 120)
    abst[f"kosong_ragu={kosong}"]["asli_≥120"] = {"alarm": float(np.isin(dr[m], (1, 2)).mean()), "tidak_yakin": float((dr[m] == 3).mean())}
abst["ambang_paket"] = q_pk
print(json.dumps(abst, indent=1))

{
 "kosong_ragu=False": {
  "ambang_median": 0.32917320728302,
  "afdb": {
   "Normal": [
    0.9238444444444445,
    0.07559595959595959,
    0.0005595959595959596,
    0.0
   ],
   "AFIB": [
    0.06498040288032085,
    0.9316956825570444,
    0.0033239145626348253,
    0.0
   ],
   "AFL": [
    0.14774305555555556,
    0.2604166666666667,
    0.5918402777777778,
    0.0
   ]
  },
  "asli_\u2265120": {
   "alarm": 0.9840109639104614,
   "tidak_yakin": 0.0
  }
 },
 "kosong_ragu=True": {
  "ambang_median": 0.32917320728302,
  "afdb": {
   "Normal": [
    0.9028101010101011,
    0.057448484848484846,
    0.0003717171717171717,
    0.03936969696969697
   ],
   "AFIB": [
    0.0429830158296114,
    0.8939689484398262,
    0.0027162519369246193,
    0.060331783793637773
   ],
   "AFL": [
    0.14166666666666666,
    0.221875,
    0.5390625,
    0.09739583333333333
   ]
  },
  "asli_\u2265120": {
   "alarm": 0.860666971219735,
   "tidak_yakin": 0.12928277752398357
  }
 },
 "ambang_paket": 0

## 4. Metrik episode v3.1 di AFDB (EC57: sensitivitas dan +P episode/durasi, episode palsu per jam)

In [8]:
zz = np.load(K.REPO / "cache/dataset_v2_detector.npz")
rec, ts = zz["records"].astype(str), zz["t_start"]
episode = {}
for kelas, n in ((1, "AFIB"), (2, "AFL")):
    ms = [K.metrik_episode(rec, ts, d32["y"], pred31[r], kelas) for r in range(5)]
    episode[n] = {m: float(np.mean([x[m] for x in ms])) for m in ms[0]}
pd.DataFrame(episode).round(3)

,AFIB,AFL
se_episode,0.970,0.686
ppv_episode,0.562,0.403
se_durasi,0.932,0.592
ppv_durasi,0.887,0.715
n_ref,881.000,14.000
n_det,6252.400,198.200
palsu_per_jam,11.858,0.521


In [9]:
ring = {"data": data_tab.to_dict(orient="index"), "alarm_pita": alarm, "utama": utama, "aturan_afdb": aturan_afdb,
        "aturan_asli": aturan_asli, "abstain": abst, "episode_v31": episode, "label": LABEL, "lingkungan": Ev._lingkungan()}
(K.CACHE / "ringkasan.json").write_text(json.dumps(ring, indent=1, default=float, ensure_ascii=False))
print("tersimpan:", K.CACHE / "ringkasan.json")

tersimpan: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v3_3/ringkasan.json


## 5. Kesimpulan

Dijalankan di PC (RTX 5060) lewat `tools/run_notebook.py`, 6 November 2026 (±9 menit dengan cache prediksi; prediksi semua model ±19 menit). Evaluasi saja; paket produksi tidak disentuh.

- **Sinus takikardia asli dialarmkan hampir selalu.** Mode perangkat, ketiga sumber, window ≥ 120 bpm (n = 2.189): paket v3.1 98,4% [97–100]; v1 98,0%; v2 99,9%. Di EPFL (bersepeda maksimal) dan Wrist: 100%. Temuan v3.2 (sintetis) terkonfirmasi di data asli.
- **Hanya model yang pernah melihat sinus takikardia yang membaik:** v8c 60,3% [53–68] (−38,1 poin, P = 1,00), v8a 63,8%, v7a 63,9% (sangat baik sampai 130 bpm, 100% di atas), v7b 70,9%. Kandidat v11 tetap 89–96%. Trade-off v7/v8 di AFDB (AFL 2:1) ada di laporan masing-masing.
- **Detektor R-peak 50 Hz bermasalah saat latihan:** EPFL hanya 68% window lolos cek (manual 95%); CV IBI median 0,028 vs 0,009 (manual). Akibatnya ada alarm AF pada detak normal saat beraktivitas: v3.1 di < 100 bpm 21% (EPHNOGRAM) dan 29% (Wrist perangkat), tetapi 9% di Wrist dengan R-peak manual.
- **Aturan produk** (≥ 110 bpm & CV ≤ 0,05) menangkap 73% (EPFL), 83% (Wrist), 61% (EPHNOGRAM) alarm ≥ 120 bpm; sisanya lolos karena jitter detektor menaikkan CV. Biaya di AFDB: 86% AFL yang terdeteksi ikut dialihkan.
- **Abstain conformal (cakupan 90%) tidak menolong:** ambang ±0,33, model yakin pada kesalahannya → 0% "tidak yakin". Bila himpunan kosong juga dianggap ragu: 12,9% sinus takikardia menjadi "tidak yakin" (alarm 86%), dengan biaya 3,9% window Normal dan 9,7% window AFL di AFDB ikut ragu.
- **Metrik episode v3.1 (gaya EC57) di AFDB:** AFL Se episode 0,686, +P 0,403, 0,52 episode palsu/jam; AFIB Se 0,970, +P 0,562, 11,9 episode palsu/jam (AF yang terpecah-pecah).
- **Tidak bisa disimpulkan:** ritme subjek latihan fisik tidak dilabeli (diasumsikan sinus); subjek muda dan sehat.